In [1]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    "goosmanlei/amazon_reviews_multi",
    "en"
)

train_df = dataset["train"].to_pandas()

print(train_df.shape)
print(train_df.columns.tolist())

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(200000, 8)
['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category']


In [2]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["text"] = train_df["review_body"].apply(clean_text)

print(train_df["text"].head())
print("\nMissing values:", train_df["text"].isna().sum())

0    arrived broken. manufacturer defect. two of th...
1    the cabinet dot were all detached from backing...
2    i received my first order of this product and ...
3    this product is a piece of shit. do not buy. d...
4    went through 3 in one day doesn't fit correct ...
Name: text, dtype: str

Missing values: 0


In [3]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    max_features=20000,
    min_df=10,
    max_df=0.90,
    stop_words="english",
    ngram_range=(1, 1)
)

X = vectorizer.fit_transform(train_df["text"])

print("Document-Term Matrix shape:", X.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

Document-Term Matrix shape: (200000, 12111)
Vocabulary size: 12111


In [4]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(
    n_components=10,
    random_state=42,
    learning_method="online",
    max_iter=10,
    batch_size=1024,
    evaluate_every=-1,
    n_jobs=-1
)

lda.fit(X)

print("LDA training completed.")
print("Number of topics:", lda.n_components)

LDA training completed.
Number of topics: 10


In [5]:
feature_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    top_words = [
        feature_names[i]
        for i in topic.argsort()[-10:][::-1]
    ]
    
    print(f"Topic {topic_idx}:")
    print(", ".join(top_words))
    print()

Topic 0:
work, works, great, light, does, charge, battery, use, doesn, did

Topic 1:
quality, expected, perfect, good, bag, beautiful, love, excellent, pair, came

Topic 2:
like, water, just, use, plastic, product, exactly, little, don, open

Topic 3:
book, like, read, money, good, don, really, just, story, know

Topic 4:
case, phone, easy, screen, good, does, cover, like, job, great

Topic 5:
use, time, bought, used, old, months, ve, just, great, worked

Topic 6:
like, size, fit, nice, small, great, good, little, color, just

Topic 7:
product, received, item, box, ordered, amazon, return, arrived, order, gift

Topic 8:
hair, soft, skin, fun, dry, product, use, head, face, like

Topic 9:
great, easy, use, love, good, sound, works, quality, dog, recommend



In [11]:
import sys

print(sys.executable)

c:\Users\sondo\AppData\Local\Programs\Python\Python312\python.exe


In [13]:
import gensim

print("Gensim version:", gensim.__version__)

Gensim version: 4.4.0


In [14]:
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

# Prepare tokenized documents
texts = train_df["text"].apply(lambda x: x.split()).tolist()

# Create dictionary
dictionary = Dictionary(texts)

# Get top words for each LDA topic
topics = [
    [
        feature_names[i]
        for i in topic.argsort()[-10:][::-1]
    ]
    for topic in lda.components_
]

# Calculate coherence
coherence_model = CoherenceModel(
    topics=topics,
    texts=texts,
    dictionary=dictionary,
    coherence="c_v"
)

coherence_score = coherence_model.get_coherence()

print("LDA Coherence Score (c_v):", coherence_score)

LDA Coherence Score (c_v): 0.44799847657487774


In [15]:
lda_5 = LatentDirichletAllocation(
    n_components=5,
    random_state=42,
    learning_method="online",
    max_iter=10,
    batch_size=1024,
    evaluate_every=-1,
    n_jobs=-1
)

lda_5.fit(X)

print("LDA with 5 topics trained.")

LDA with 5 topics trained.


In [16]:
topics_5 = [
    [
        feature_names[i]
        for i in topic.argsort()[-10:][::-1]
    ]
    for topic in lda_5.components_
]

coherence_model_5 = CoherenceModel(
    topics=topics_5,
    texts=texts,
    dictionary=dictionary,
    coherence="c_v"
)

coherence_score_5 = coherence_model_5.get_coherence()

print("LDA Coherence Score (5 topics):", coherence_score_5)

LDA Coherence Score (5 topics): 0.44484343564825296


In [17]:

lda_15 = LatentDirichletAllocation(
    n_components=15,
    random_state=42,
    learning_method="online",
    max_iter=10,
    batch_size=1024,
    evaluate_every=-1,
    n_jobs=-1
)

lda_15.fit(X)

print("LDA with 15 topics trained.")

LDA with 15 topics trained.


In [18]:
topics_15 = [
    [
        feature_names[i]
        for i in topic.argsort()[-10:][::-1]
    ]
    for topic in lda_15.components_
]

coherence_model_15 = CoherenceModel(
    topics=topics_15,
    texts=texts,
    dictionary=dictionary,
    coherence="c_v"
)

coherence_score_15 = coherence_model_15.get_coherence()

print("LDA Coherence Score (15 topics):", coherence_score_15)

LDA Coherence Score (15 topics): 0.4278454743578644


In [19]:
lda_20 = LatentDirichletAllocation(
    n_components=20,
    random_state=42,
    learning_method="online",
    max_iter=10,
    batch_size=1024,
    evaluate_every=-1,
    n_jobs=-1
)

lda_20.fit(X)

print("LDA with 20 topics trained.")

LDA with 20 topics trained.


In [20]:
topics_20 = [
    [
        feature_names[i]
        for i in topic.argsort()[-10:][::-1]
    ]
    for topic in lda_20.components_
]

coherence_model_20 = CoherenceModel(
    topics=topics_20,
    texts=texts,
    dictionary=dictionary,
    coherence="c_v"
)

coherence_score_20 = coherence_model_20.get_coherence()

print("LDA Coherence Score (20 topics):", coherence_score_20)

LDA Coherence Score (20 topics): 0.4464241948506036


In [21]:
custom_stopwords = {
    "good", "great", "like", "just",
    "use", "used", "using",
    "product", "products",
    "really", "nice", "little"
}

texts_clean = [
    [
        word
        for word in text.split()
        if word not in custom_stopwords
    ]
    for text in train_df["text"]
]

print("Number of documents:", len(texts_clean))
print("Example:", texts_clean[0][:30])

Number of documents: 200000
Example: ['arrived', 'broken.', 'manufacturer', 'defect.', 'two', 'of', 'the', 'legs', 'of', 'the', 'base', 'were', 'not', 'completely', 'formed,', 'so', 'there', 'was', 'no', 'way', 'to', 'insert', 'the', 'casters.', 'i', 'unpackaged', 'the', 'entire', 'chair', 'and']


In [22]:

import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stopwords = {
    "good", "great", "like", "just",
    "use", "used", "using",
    "product", "products",
    "really", "nice", "little"
}

all_stopwords = set(ENGLISH_STOP_WORDS) | custom_stopwords

texts_clean = []

for text in train_df["text"]:
    words = re.findall(r"\b[a-z]{2,}\b", text.lower())
    
    words = [
        word
        for word in words
        if word not in all_stopwords
    ]
    
    texts_clean.append(words)

print("Number of documents:", len(texts_clean))
print("Example:", texts_clean[0][:30])

Number of documents: 200000
Example: ['arrived', 'broken', 'manufacturer', 'defect', 'legs', 'base', 'completely', 'formed', 'way', 'insert', 'casters', 'unpackaged', 'entire', 'chair', 'hardware', 'noticing', 'll', 'spend', 'twice', 'time', 'boxing', 'useless', 'thing', 'send', 'star', 'review', 'chair', 'got', 'sit', 'far']


In [23]:
vectorizer_clean = CountVectorizer(
    max_features=20000,
    min_df=10,
    max_df=0.90,
    ngram_range=(1, 1)
)

X_clean = vectorizer_clean.fit_transform(
    [" ".join(words) for words in texts_clean]
)

print("Clean Document-Term Matrix shape:", X_clean.shape)
print("Clean vocabulary size:", len(vectorizer_clean.vocabulary_))

Clean Document-Term Matrix shape: (200000, 11842)
Clean vocabulary size: 11842


In [25]:
lda_clean_10 = LatentDirichletAllocation(
    n_components=10,
    random_state=42,
    learning_method="online",
    max_iter=10,
    batch_size=1024,
    evaluate_every=-1,
    n_jobs=-1
)

lda_clean_10.fit(X_clean)

print("Clean LDA with 10 topics trained.")

Clean LDA with 10 topics trained.


In [26]:
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

# Build dictionary from cleaned texts
dictionary_clean = Dictionary(texts_clean)

# Extract top words for each topic
feature_names_clean = vectorizer_clean.get_feature_names_out()

topics_clean = [
    [feature_names_clean[i] for i in topic.argsort()[-10:][::-1]]
    for topic in lda_clean_10.components_
]

# Calculate coherence
coherence_model_clean = CoherenceModel(
    topics=topics_clean,
    texts=texts_clean,
    dictionary=dictionary_clean,
    coherence="c_v"
)

coherence_score_clean = coherence_model_clean.get_coherence()

print("Improved LDA Coherence Score:", coherence_score_clean)

Improved LDA Coherence Score: 0.5776141208790428


In [27]:
for topic_idx, topic in enumerate(topics_clean):
    print(f"\nTopic {topic_idx}:")
    print(", ".join(topic))


Topic 0:
color, love, hair, work, colors, look, does, ok, job, feel

Topic 1:
perfect, easy, love, exactly, set, loves, perfectly, inside, described, hand

Topic 2:
cute, overall, strong, sturdy, dog, pretty, larger, bit, metal, characters

Topic 3:
works, work, easy, light, does, battery, phone, doesn, charge, sound

Topic 4:
box, received, item, ordered, came, arrived, amazon, return, order, gift

Topic 5:
book, read, story, reading, life, books, know, expect, series, delivered

Topic 6:
quality, price, better, expected, soft, smell, awesome, bad, high, best

Topic 7:
time, money, old, months, ve, bought, worked, year, day, working

Topic 8:
case, phone, screen, comfortable, cover, looks, easy, beautiful, easily, plastic

Topic 9:
size, small, fit, love, water, wear, bag, big, large, perfect


In [28]:
# Additional domain-specific stopwords
refined_stopwords = {
    "good", "great", "like", "just",
    "use", "used", "using",
    "product", "products",
    "really", "nice", "little",
    "perfect", "perfectly",
    "love", "loves",
    "easy", "easily",
    "better", "best",
    "awesome",
    "ok", "overall",
    "work", "works"
}

all_stopwords_refined = set(ENGLISH_STOP_WORDS) | refined_stopwords

texts_refined = []

for text in train_df["text"]:
    words = re.findall(r"\b[a-z]{2,}\b", text.lower())
    words = [
        word
        for word in words
        if word not in all_stopwords_refined
    ]
    texts_refined.append(words)

print("Number of documents:", len(texts_refined))
print("Example:", texts_refined[0][:30])

Number of documents: 200000
Example: ['arrived', 'broken', 'manufacturer', 'defect', 'legs', 'base', 'completely', 'formed', 'way', 'insert', 'casters', 'unpackaged', 'entire', 'chair', 'hardware', 'noticing', 'll', 'spend', 'twice', 'time', 'boxing', 'useless', 'thing', 'send', 'star', 'review', 'chair', 'got', 'sit', 'far']


In [29]:
vectorizer_refined = CountVectorizer(
    max_features=20000,
    min_df=10,
    max_df=0.90,
    ngram_range=(1, 1)
)

X_refined = vectorizer_refined.fit_transform(
    [" ".join(words) for words in texts_refined]
)

print("Refined Document-Term Matrix shape:", X_refined.shape)
print("Refined vocabulary size:", len(vectorizer_refined.vocabulary_))

Refined Document-Term Matrix shape: (200000, 11829)
Refined vocabulary size: 11829


In [30]:
lda_refined_10 = LatentDirichletAllocation(
    n_components=10,
    random_state=42,
    learning_method="online",
    max_iter=10,
    batch_size=1024,
    evaluate_every=-1,
    n_jobs=-1
)

lda_refined_10.fit(X_refined)

print("Refined LDA with 10 topics trained.")

Refined LDA with 10 topics trained.


In [31]:
dictionary_refined = Dictionary(texts_refined)

feature_names_refined = vectorizer_refined.get_feature_names_out()

topics_refined = [
    [feature_names_refined[i] for i in topic.argsort()[-10:][::-1]]
    for topic in lda_refined_10.components_
]

coherence_model_refined = CoherenceModel(
    topics=topics_refined,
    texts=texts_refined,
    dictionary=dictionary_refined,
    coherence="c_v"
)

coherence_score_refined = coherence_model_refined.get_coherence()

print("Refined LDA Coherence Score:", coherence_score_refined)

Refined LDA Coherence Score: 0.5702398854427011


In [32]:
vectorizer_bigram = CountVectorizer(
    max_features=30000,
    min_df=10,
    max_df=0.90,
    ngram_range=(1, 2)
)

X_bigram = vectorizer_bigram.fit_transform(
    [" ".join(words) for words in texts_clean]
)

print("Bigram Document-Term Matrix shape:", X_bigram.shape)
print("Bigram vocabulary size:", len(vectorizer_bigram.vocabulary_))

Bigram Document-Term Matrix shape: (200000, 30000)
Bigram vocabulary size: 30000


In [33]:
lda_bigram_10 = LatentDirichletAllocation(
    n_components=10,
    random_state=42,
    learning_method="online",
    max_iter=10,
    batch_size=1024,
    evaluate_every=-1,
    n_jobs=-1
)

lda_bigram_10.fit(X_bigram)

print("Bigram LDA with 10 topics trained.")

Bigram LDA with 10 topics trained.


In [34]:
dictionary_bigram = Dictionary(texts_clean)

feature_names_bigram = vectorizer_bigram.get_feature_names_out()

topics_bigram = [
    [feature_names_bigram[i] for i in topic.argsort()[-10:][::-1]]
    for topic in lda_bigram_10.components_
]

coherence_model_bigram = CoherenceModel(
    topics=topics_bigram,
    texts=texts_clean,
    dictionary=dictionary_bigram,
    coherence="c_v"
)

coherence_score_bigram = coherence_model_bigram.get_coherence()

print("Bigram LDA Coherence Score:", coherence_score_bigram)

Bigram LDA Coherence Score: 0.5686643736084924


In [35]:
lda_clean_5 = LatentDirichletAllocation(
    n_components=5,
    random_state=42,
    learning_method="online",
    max_iter=10,
    batch_size=1024,
    evaluate_every=-1,
    n_jobs=-1
)

lda_clean_5.fit(X_clean)

print("Clean LDA with 5 topics trained.")

Clean LDA with 5 topics trained.


In [36]:
feature_names_clean = vectorizer_clean.get_feature_names_out()

topics_clean_5 = [
    [feature_names_clean[i] for i in topic.argsort()[-10:][::-1]]
    for topic in lda_clean_5.components_
]

coherence_model_clean_5 = CoherenceModel(
    topics=topics_clean_5,
    texts=texts_clean,
    dictionary=dictionary_clean,
    coherence="c_v"
)

coherence_score_clean_5 = coherence_model_clean_5.get_coherence()

print("Clean LDA (K=5) Coherence Score:", coherence_score_clean_5)

Clean LDA (K=5) Coherence Score: 0.5672488098435787


In [37]:
import os
import joblib

lda_path = "../../models/topics"

os.makedirs(lda_path, exist_ok=True)

joblib.dump(
    lda_clean_10,
    os.path.join(lda_path, "lda_clean_10.joblib")
)

joblib.dump(
    vectorizer_clean,
    os.path.join(lda_path, "count_vectorizer_clean.joblib")
)

joblib.dump(
    topics_clean,
    os.path.join(lda_path, "lda_topics_clean.joblib")
)

print("LDA model saved successfully.")
print("Vectorizer saved successfully.")
print("Topics saved successfully.")

LDA model saved successfully.
Vectorizer saved successfully.
Topics saved successfully.


In [38]:
import json
import os

results = {
    "baseline_lda_k10": 0.44799847657487774,
    "improved_preprocessing_k10": 0.5776141208790428,
    "refined_stopwords_k10": 0.5702398854427011,
    "bigram_lda_k10": 0.5686643736084924,
    "improved_preprocessing_k5": 0.5672488098435787
}

results_path = "../../models/topics/lda_experiments.json"

with open(results_path, "w") as f:
    json.dump(results, f, indent=4)

print("LDA experiment results saved successfully.")
print(results_path)

LDA experiment results saved successfully.
../../models/topics/lda_experiments.json


In [39]:
import os

files = [
    "../../models/topics/lda_clean_10.joblib",
    "../../models/topics/count_vectorizer_clean.joblib",
    "../../models/topics/lda_topics_clean.joblib",
    "../../models/topics/lda_experiments.json"
]

print("LDA files verification:\n")

all_ok = True

for file_path in files:
    exists = os.path.exists(file_path)
    size_kb = os.path.getsize(file_path) / 1024 if exists else 0

    status = "OK" if exists and size_kb > 0 else "MISSING / EMPTY"

    print(f"{status:16} | {size_kb:,.2f} KB | {file_path}")

    if not exists or size_kb == 0:
        all_ok = False

print("\nFinal status:", "ALL FILES SAVED ✅" if all_ok else "CHECK FILES ⚠️")

LDA files verification:

OK               | 1,856.47 KB | ../../models/topics/lda_clean_10.joblib
OK               | 333.10 KB | ../../models/topics/count_vectorizer_clean.joblib
OK               | 0.81 KB | ../../models/topics/lda_topics_clean.joblib
OK               | 0.25 KB | ../../models/topics/lda_experiments.json

Final status: ALL FILES SAVED ✅


In [40]:
import os

paths_to_check = {
    "Emotion BERT": "../../models/emotion/bert_emotion",
    "Sentiment Binary BERT": "../../models/sentiment/bert_binary_weighted",
    "Sentiment 3-Class": "../../models/sentiment",
    "LDA": "../../models/topics",
}

print("========== MODEL STORAGE CHECK ==========\n")

for name, path in paths_to_check.items():
    exists = os.path.exists(path)

    if exists:
        files = []
        for root, dirs, filenames in os.walk(path):
            for filename in filenames:
                files.append(os.path.join(root, filename))

        print(f"✅ {name}")
        print(f"   Path: {path}")
        print(f"   Files found: {len(files)}")

        for f in files[:10]:
            size_mb = os.path.getsize(f) / (1024 * 1024)
            print(f"      - {os.path.basename(f)} ({size_mb:.2f} MB)")

        if len(files) > 10:
            print(f"      ... and {len(files) - 10} more files")

        print()
    else:
        print(f"❌ {name} NOT FOUND")
        print(f"   Path: {path}\n")

========== MODEL STORAGE CHECK ==========

✅ Emotion BERT
   Path: ../../models/emotion/bert_emotion
   Files found: 4
      - config.json (0.00 MB)
      - model.safetensors (417.68 MB)
      - tokenizer.json (0.68 MB)
      - tokenizer_config.json (0.00 MB)

❌ Sentiment Binary BERT NOT FOUND
   Path: ../../models/sentiment/bert_binary_weighted

✅ Sentiment 3-Class
   Path: ../../models/sentiment
   Files found: 0

✅ LDA
   Path: ../../models/topics
   Files found: 4
      - count_vectorizer_clean.joblib (0.33 MB)
      - lda_clean_10.joblib (1.81 MB)
      - lda_experiments.json (0.00 MB)
      - lda_topics_clean.joblib (0.00 MB)



In [41]:
import os

project_root = os.path.abspath("../..")

print("Searching for sentiment model files...\n")

matches = []

for root, dirs, files in os.walk(project_root):
    # Skip unnecessary folders
    dirs[:] = [
        d for d in dirs
        if d not in {".git", ".venv", "__pycache__", ".ipynb_checkpoints"}
    ]

    for file in files:
        if file in {
            "model.safetensors",
            "pytorch_model.bin",
            "config.json",
            "tokenizer.json"
        }:
            full_path = os.path.join(root, file)
            matches.append(full_path)

if matches:
    for path in matches:
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{size_mb:8.2f} MB | {path}")
else:
    print("No model files found.")

print(f"\nTotal model-related files found: {len(matches)}")

Searching for sentiment model files...

    0.00 MB | d:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\models\emotion\bert_emotion\config.json
  417.68 MB | d:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\models\emotion\bert_emotion\model.safetensors
    0.68 MB | d:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\models\emotion\bert_emotion\tokenizer.json
    0.00 MB | d:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\notebooks\02_sentiment\bert_sentiment_binary_weighted_results\checkpoint-4000\config.json
  417.67 MB | d:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\notebooks\02_sentiment\bert_sentiment_binary_weighted_results\checkpoint-4000\model.safetensors
    0.68 MB | d:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\notebooks\02_sentiment\bert_sentiment_binary_weighted_results\checkpoint-4000\tokenizer.json
    0.00 MB | d:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\notebooks\02_sentiment\bert_sentiment_binary_weighted_results\

In [42]:
import os
import shutil

project_root = os.path.abspath("../..")

source_binary = os.path.join(
    project_root,
    "notebooks",
    "02_sentiment",
    "models",
    "sentiment",
    "bert_binary_weighted"
)

source_weighted = os.path.join(
    project_root,
    "notebooks",
    "02_sentiment",
    "models",
    "sentiment",
    "bert_weighted"
)

target_dir = os.path.join(
    project_root,
    "models",
    "sentiment"
)

os.makedirs(target_dir, exist_ok=True)

# Copy Binary model
shutil.copytree(
    source_binary,
    os.path.join(target_dir, "bert_binary_weighted"),
    dirs_exist_ok=True
)

# Copy 3-class weighted model
shutil.copytree(
    source_weighted,
    os.path.join(target_dir, "bert_weighted"),
    dirs_exist_ok=True
)

print("Sentiment models organized successfully. ✅")
print("\nFinal locations:")
print(os.path.join(target_dir, "bert_binary_weighted"))
print(os.path.join(target_dir, "bert_weighted"))

Sentiment models organized successfully. ✅

Final locations:
d:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\models\sentiment\bert_binary_weighted
d:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\models\sentiment\bert_weighted


In [43]:
import os

project_root = os.path.abspath("../..")

models_to_check = {
    "Sentiment - Binary Weighted":
        os.path.join(project_root, "models", "sentiment", "bert_binary_weighted"),

    "Sentiment - 3-Class Weighted":
        os.path.join(project_root, "models", "sentiment", "bert_weighted"),

    "Emotion - BERT":
        os.path.join(project_root, "models", "emotion", "bert_emotion"),

    "Topics - LDA":
        os.path.join(project_root, "models", "topics")
}

required_files = {
    "Sentiment - Binary Weighted":
        ["config.json", "model.safetensors", "tokenizer.json"],

    "Sentiment - 3-Class Weighted":
        ["config.json", "model.safetensors", "tokenizer.json"],

    "Emotion - BERT":
        ["config.json", "model.safetensors", "tokenizer.json"],

    "Topics - LDA":
        [
            "lda_clean_10.joblib",
            "count_vectorizer_clean.joblib",
            "lda_topics_clean.joblib",
            "lda_experiments.json"
        ]
}

print("=" * 60)
print("FINAL MODEL STORAGE VERIFICATION")
print("=" * 60)

all_ok = True

for name, folder in models_to_check.items():
    print(f"\n{name}")

    for filename in required_files[name]:
        path = os.path.join(folder, filename)

        if os.path.isfile(path) and os.path.getsize(path) > 0:
            size_mb = os.path.getsize(path) / (1024 * 1024)
            print(f"  ✅ {filename} ({size_mb:.2f} MB)")
        else:
            print(f"  ❌ {filename} MISSING / EMPTY")
            all_ok = False

print("\n" + "=" * 60)

if all_ok:
    print("🎉 ALL FINAL MODELS ARE SAVED AND VERIFIED.")
    print("You can safely close the laptop.")
else:
    print("⚠️ SOME FILES NEED ATTENTION.")

FINAL MODEL STORAGE VERIFICATION

Sentiment - Binary Weighted
  ✅ config.json (0.00 MB)
  ✅ model.safetensors (417.67 MB)
  ✅ tokenizer.json (0.68 MB)

Sentiment - 3-Class Weighted
  ✅ config.json (0.00 MB)
  ✅ model.safetensors (417.67 MB)
  ✅ tokenizer.json (0.68 MB)

Emotion - BERT
  ✅ config.json (0.00 MB)
  ✅ model.safetensors (417.68 MB)
  ✅ tokenizer.json (0.68 MB)

Topics - LDA
  ✅ lda_clean_10.joblib (1.81 MB)
  ✅ count_vectorizer_clean.joblib (0.33 MB)
  ✅ lda_topics_clean.joblib (0.00 MB)
  ✅ lda_experiments.json (0.00 MB)

🎉 ALL FINAL MODELS ARE SAVED AND VERIFIED.
You can safely close the laptop.
